# 🧠 Klasifikasi Personality (Extrovert / Introvert) — Random Forest

**UTS Machine Learning — Diaz**

Notebook ini melakukan klasifikasi kepribadian (`Extrovert` vs `Introvert`) menggunakan `RandomForestClassifier` dari scikit-learn.

**Pipeline:**
1. Load & EDA singkat
2. Cleaning & Preprocessing
3. Hyperparameter tuning (`n_estimators`, `max_depth`)
4. Evaluasi model terbaik (Accuracy, Precision, Recall, F1, Confusion Matrix, Feature Importance)
5. Simpan pipeline & metadata

---
## 0 · Setup & Load Dataset

In [ ]:
import os, json, warnings, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

print('✅ Libraries loaded.')

In [ ]:
# ---------------------------------------------------------------------------
# Strategi load dataset:
#   1. Coba path lokal relatif  (jalan di repo / local Jupyter)
#   2. Coba Google Drive mount  (jalan di Colab + Drive)
#   3. Fallback: upload manual  (jalan di Colab tanpa Drive)
# ---------------------------------------------------------------------------

DATASET_FILENAME = 'personality_datasert.csv'

# Path lokal (relatif dari notebooks/)
LOCAL_PATH = os.path.join('..', 'dataset', DATASET_FILENAME)

# Path Google Drive (sesuaikan folder jika berbeda)
DRIVE_PATH = f'/content/drive/MyDrive/dataset/{DATASET_FILENAME}'

df = None

if os.path.isfile(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f'📂 Dataset dimuat dari path lokal: {LOCAL_PATH}')
else:
    # Cek apakah berjalan di Google Colab
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

    if IN_COLAB:
        # Coba mount Google Drive
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            if os.path.isfile(DRIVE_PATH):
                df = pd.read_csv(DRIVE_PATH)
                print(f'📂 Dataset dimuat dari Google Drive: {DRIVE_PATH}')
        except Exception:
            pass

        # Fallback: upload manual
        if df is None:
            from google.colab import files
            print(f'⬆️  Upload file "{DATASET_FILENAME}" secara manual:')
            uploaded = files.upload()
            df = pd.read_csv(DATASET_FILENAME)
            print(f'📂 Dataset dimuat dari upload manual.')
    else:
        raise FileNotFoundError(
            f'Dataset tidak ditemukan di "{LOCAL_PATH}". '
            f'Pastikan file berada di folder dataset/.'
        )

print(f'\n✅ Dataset berhasil dimuat — {df.shape[0]} baris, {df.shape[1]} kolom.')

---
## 1 · Exploratory Data Analysis (EDA)

In [ ]:
# 1a. Shape & dtypes
print('='*50)
print('SHAPE:', df.shape)
print('='*50)
print('\nDTYPES:')
print(df.dtypes)
print('\n')
df.head(10)

In [ ]:
# 1b. Missing values & duplicates
print('='*50)
print('MISSING VALUES (isnull().sum()):')
print('='*50)
print(df.isnull().sum())
print(f'\nTotal missing : {df.isnull().sum().sum()}')
print(f'Total duplikat: {df.duplicated().sum()}')

In [ ]:
# 1c. Statistik deskriptif
df.describe(include='all').T

In [ ]:
# 1d. Distribusi target (Personality)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
order = ['Introvert', 'Extrovert']
palette = {'Introvert': '#5B8FF9', 'Extrovert': '#F6BD16'}
sns.countplot(data=df, x='Personality', order=order, palette=palette, ax=axes[0])
axes[0].set_title('Distribusi Target — Personality')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Pie chart
counts = df['Personality'].value_counts().reindex(order)
axes[1].pie(counts, labels=order, autopct='%1.1f%%',
            colors=[palette[k] for k in order], startangle=90)
axes[1].set_title('Proporsi Target')

plt.tight_layout()
plt.show()

---
## 2 · Cleaning & Preprocessing

In [ ]:
# 2a. Identifikasi kolom numerik & kategorikal (tanpa target)
TARGET = 'Personality'
LABEL_MAP = {'Introvert': 0, 'Extrovert': 1}

num_cols = df.drop(columns=[TARGET]).select_dtypes(include='number').columns.tolist()
cat_cols = df.drop(columns=[TARGET]).select_dtypes(include='object').columns.tolist()

print(f'Numerik   ({len(num_cols)}): {num_cols}')
print(f'Kategori  ({len(cat_cols)}): {cat_cols}')

In [ ]:
# 2b. Imputasi
#   - Numerik  → median
#   - Kategori → modus

for col in num_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f'  Imputasi median — {col}: {median_val}')

for col in cat_cols:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f'  Imputasi modus  — {col}: {mode_val}')

# Target
if df[TARGET].isnull().sum() > 0:
    mode_val = df[TARGET].mode()[0]
    df[TARGET].fillna(mode_val, inplace=True)
    print(f'  Imputasi modus  — {TARGET}: {mode_val}')

print(f'\n✅ Missing values setelah imputasi: {df.isnull().sum().sum()}')

In [ ]:
# 2c. Hapus duplikat
n_before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
n_after = len(df)
print(f'Duplikat dihapus: {n_before - n_after}  |  Baris tersisa: {n_after}')

In [ ]:
# 2d. Encode target
df['Personality_encoded'] = df[TARGET].map(LABEL_MAP)
print('Label mapping:', LABEL_MAP)
print(df[['Personality', 'Personality_encoded']].value_counts())

In [ ]:
# 2e. Simpan metadata fitur SEBELUM split (untuk disimpan nanti)
feature_meta = {}

for col in num_cols:
    feature_meta[col] = {
        'type': 'numerical',
        'min': float(df[col].min()),
        'max': float(df[col].max()),
        'median': float(df[col].median())
    }

for col in cat_cols:
    feature_meta[col] = {
        'type': 'categorical',
        'categories': sorted(df[col].unique().tolist())
    }

# Rata-rata fitur numerik per kelas
mean_per_class = {}
for label_name, label_val in LABEL_MAP.items():
    subset = df[df['Personality_encoded'] == label_val][num_cols]
    mean_per_class[label_name] = {col: round(float(subset[col].mean()), 4) for col in num_cols}

print('✅ Feature metadata disimpan.')

In [ ]:
# 2f. Train-Test Split (80/20, stratify, random_state=42)
X = df[num_cols + cat_cols]
y = df['Personality_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train distribusi:\n{y_train.value_counts()}')
print(f'y_test  distribusi:\n{y_test.value_counts()}')

In [ ]:
# 2g. ColumnTransformer (OneHotEncoder untuk kategorikal, passthrough untuk numerik)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ],
    remainder='drop'
)

print('✅ ColumnTransformer siap.')
print(f'   Kategorikal → OneHotEncoder: {cat_cols}')
print(f'   Numerik     → passthrough  : {num_cols}')

---
## 3 · Hyperparameter Tuning & Training

In [ ]:
# 3a. Grid search manual: n_estimators × max_depth
n_estimators_list = [50, 100, 150, 200, 300]
max_depth_list    = [None, 5, 10, 15, 20]

results = []

print(f'🔍 Menguji {len(n_estimators_list) * len(max_depth_list)} kombinasi ...\n')

for n_est in n_estimators_list:
    for depth in max_depth_list:
        pipe = Pipeline([
            ('preprocessor', preprocessor),
            ('clf', RandomForestClassifier(
                n_estimators=n_est,
                max_depth=depth,
                random_state=42,
                n_jobs=-1
            ))
        ])
        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec  = recall_score(y_test, y_pred)
        f1   = f1_score(y_test, y_pred)

        results.append({
            'n_estimators': n_est,
            'max_depth': depth,
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1_score': f1,
            'pipeline': pipe
        })

print('✅ Semua kombinasi selesai diuji.')

In [ ]:
# 3b. Tabel hasil — urutkan berdasarkan F1 Score
results_df = pd.DataFrame(results).drop(columns=['pipeline'])
results_df = results_df.sort_values('f1_score', ascending=False).reset_index(drop=True)
results_df.index += 1
results_df.index.name = 'Rank'

# Format persentase
styled = results_df.style.format({
    'accuracy':  '{:.4f}',
    'precision': '{:.4f}',
    'recall':    '{:.4f}',
    'f1_score':  '{:.4f}'
}).background_gradient(subset=['f1_score'], cmap='YlGn')

styled

In [ ]:
# 3c. Pilih model terbaik (F1 tertinggi)
best = max(results, key=lambda r: r['f1_score'])
best_pipeline = best['pipeline']

print('🏆 MODEL TERBAIK')
print(f"   n_estimators : {best['n_estimators']}")
print(f"   max_depth    : {best['max_depth']}")
print(f"   Accuracy     : {best['accuracy']:.4f}")
print(f"   Precision    : {best['precision']:.4f}")
print(f"   Recall       : {best['recall']:.4f}")
print(f"   F1 Score     : {best['f1_score']:.4f}")

---
## 4 · Evaluasi Model Terbaik

In [ ]:
# 4a. Classification Report
y_pred_best = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_best, target_names=['Introvert (0)', 'Extrovert (1)']))

In [ ]:
# 4b. Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Introvert', 'Extrovert'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Confusion Matrix — Best Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# 4c. Feature Importance
clf_best = best_pipeline.named_steps['clf']
pre_best = best_pipeline.named_steps['preprocessor']

# Dapatkan nama fitur setelah transformasi
try:
    feature_names = pre_best.get_feature_names_out()
except AttributeError:
    # Fallback untuk sklearn < 1.0
    ohe = pre_best.named_transformers_['cat']
    cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
    feature_names = cat_feature_names + num_cols

importances = clf_best.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(feat_imp_df.to_string(index=False))

In [ ]:
# 4d. Plot Feature Importance
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=feat_imp_df, x='importance', y='feature',
            palette='viridis', ax=ax)
ax.set_title('Feature Importance — Best Random Forest')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

---
## 5 · Simpan Pipeline & Metadata

In [ ]:
# 5a. Pastikan folder model/ ada
MODEL_DIR = os.path.join('..', 'model')
os.makedirs(MODEL_DIR, exist_ok=True)

# 5b. Simpan pipeline (.pkl)
MODEL_PATH = os.path.join(MODEL_DIR, 'personality_rf_model.pkl')
joblib.dump(best_pipeline, MODEL_PATH)
print(f'💾 Pipeline disimpan ke: {MODEL_PATH}')

In [ ]:
# 5c. Bangun & simpan metadata JSON
metadata = {
    'model': 'RandomForestClassifier',
    'best_hyperparameters': {
        'n_estimators': best['n_estimators'],
        'max_depth': best['max_depth']
    },
    'label_mapping': LABEL_MAP,
    'features': {
        'numerical': num_cols,
        'categorical': cat_cols
    },
    'feature_metadata': feature_meta,
    'mean_per_class': mean_per_class,
    'feature_importance': [
        {'feature': row['feature'], 'importance': round(float(row['importance']), 6)}
        for _, row in feat_imp_df.iterrows()
    ],
    'evaluation_metrics': {
        'accuracy':  round(float(best['accuracy']),  4),
        'precision': round(float(best['precision']), 4),
        'recall':    round(float(best['recall']),    4),
        'f1_score':  round(float(best['f1_score']),  4)
    },
    'confusion_matrix': cm.tolist(),
    'train_test_split': {
        'test_size': 0.2,
        'stratify': True,
        'random_state': 42
    },
    'dataset': DATASET_FILENAME,
    'preprocessing': {
        'imputation_numerical': 'median',
        'imputation_categorical': 'mode',
        'duplicates_removed': True,
        'column_transformer': {
            'categorical': "OneHotEncoder(drop='if_binary', handle_unknown='ignore')",
            'numerical': 'passthrough'
        }
    }
}

META_PATH = os.path.join(MODEL_DIR, 'model_metadata.json')
with open(META_PATH, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'📄 Metadata disimpan ke: {META_PATH}')

In [ ]:
# 5d. Tampilkan ringkasan metadata
print(json.dumps(metadata, indent=2, ensure_ascii=False))

In [ ]:
print('\n' + '='*60)
print('🎉  SELESAI — Semua output tersimpan di folder model/')
print('='*60)
print(f'   📦 Pipeline : {MODEL_PATH}')
print(f'   📄 Metadata : {META_PATH}')